## Building Block Layer

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [2]:
llm = ChatOpenAI(model="gpt-4.1-nano-2025-04-14")
message = "In 1 sentence, what does it mean for an AI agent to be autonomous"
reply = llm.invoke(message)
print(reply.content)

An AI agent is autonomous when it can independently perceive its environment, make decisions, and execute actions without requiring direct human intervention.


In [3]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

Silent minds weave pathways unknown,  
Guided by code, they venture alone.

In [4]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]
print(llm.invoke(messages).content)

The capital of France is Paris.


In [5]:
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content":  "What is the capital of France?"}
]

print(llm.invoke(messages_as_dicts).content)

The capital of France is Paris.


In [6]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print(get_share_price.name)
print(get_share_price.description)
print(get_share_price.args)
print(get_share_price.invoke({"symbol": "AAPL"}))

get_share_price
Return the current share price for a given ticker symbol.
{'symbol': {'title': 'Symbol', 'type': 'string'}}
241.5


In [12]:
llm_with_tools = llm.bind_tools([get_share_price])
response = llm_with_tools.invoke("What is the share price of Amazon?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_QDVWTd3bBRmU9q5aJH2QiLsJ', 'type': 'tool_call'}]


In [14]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call['name'] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

The current share price of Amazon is $198.0.


In [22]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon' ticker='AMZN' founded_year=1994
Just the ticker: AMZN


In [21]:
structured_llm

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain-openai': '1.6.0'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x76f0aa9965d0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x76f0a992d160>, root_client=<openai.OpenAI object at 0x76f0aa4633e0>, root_async_client=<openai.AsyncOpenAI object at 0x76f0a9d7ca40>, model_name='gpt-4.1-nano-2025-04-14', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Company'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Company', 'description': '', 'parameters': {'properties': {'name': {'description': 'The company name', 'type': 'string'}, 'ticker': {'description': 'The stock ticker symbol', 'type': 'string'}, 'founded_year': {'description': 'Th